# Lab3 - material de apoio

Para usar tabelas do esquema "lodlog_dw", faça:

```sql
USE lodlog_dw;
```

Para usar tabelas do esquema "lodlog_op", faça:

```sql
USE lodlog_op;
```

Caso for misturar, lembre-se de adicionar os nomes dos esquemas antes dos nomes da tabelas, ex.:

Para o DW:
```sql
lodlog_dw.dim_cat_atraso
```

Para o operacional:
```sql
lodlog_op.cliente
```

In [0]:
%sql
USE lodlog_dw;
SHOW TABLES;

In [0]:
%sql
USE lodlog_dw;
-- Query 1: Volume de entregas e atrasos por mês
SELECT DATE_TRUNC('month', data_entrega) AS mes,
       COUNT(*) AS total_entregas,
       SUM(CASE WHEN minutos_atraso > 0 THEN 1 ELSE 0 END) AS entregas_atrasadas
FROM fato_entregas
GROUP BY 1
ORDER BY 1;

In [0]:
%sql
USE lodlog_dw;
-- Query 2: Ranking de Centros de Distribuição (CD) por eficiência operacional
SELECT cd.codigo AS cd_origem,
       ROUND(AVG(fe.minutos_atraso), 2) AS atraso_medio_min,
       ROUND(SUM(fe.custo_combustivel + fe.custo_pedagio), 2) AS custo_total_operacional,
       COUNT(DISTINCT fe.sk_veiculo) AS frota_alocada
FROM fato_entregas fe
INNER JOIN dim_centro_distribuicao cd ON fe.sk_cd_origem = cd.sk_cd
WHERE fe.data_entrega >= CURRENT_DATE - INTERVAL '90 days'
GROUP BY cd.codigo
ORDER BY atraso_medio_min ASC;

In [0]:
%sql
USE lodlog_dw;
-- Query 3: Correlação estatística simples entre a distância percorrida e o custo de combustível
SELECT corr(distancia_km, custo_combustivel) AS correlacao_distancia_custo
FROM fato_entregas
WHERE data_entrega >= CURRENT_DATE - INTERVAL '30 days';

In [0]:
%sql
USE lodlog_dw;
CREATE OR REPLACE VIEW lodlog_dw.vw_features_ml AS
WITH hist_cliente AS (
    SELECT sk_cliente, data_entrega,
           AVG(minutos_atraso) OVER (
             PARTITION BY sk_cliente 
             ORDER BY data_entrega
             ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
           ) AS hist_7d
    FROM fato_entregas
),
manut_veiculo AS (
    -- Dica: Encontre a data de conclusão da última manutenção de cada veículo
    SELECT veiculo_id,
           data_conclusao,
           ROW_NUMBER() OVER (PARTITION BY veiculo_id ORDER BY data_conclusao DESC) as rn
    FROM lodlog_op.manutencao
    WHERE status = 'Concluída'
)
SELECT
    fe.sk_entrega,
    fe.distancia_km,
    fe.peso_total_kg,
    hour(fe.hora_saida) AS hora_saida,
    dayofweek(fe.data_entrega) AS dia_semana,
    dt.flag_feriado AS eh_feriado,
    fe.temperatura_media_motor,
    ROUND(h.hist_7d, 2) AS historico_atrasos_7d,
    DATEDIFF(fe.data_entrega, mv.data_conclusao) AS dias_ultima_manutencao,
    fe.indicador_atraso AS target_atraso
FROM fato_entregas fe
INNER JOIN dim_tempo dt ON fe.sk_data = dt.sk_tempo
LEFT JOIN hist_cliente h ON fe.sk_cliente = h.sk_cliente AND fe.data_entrega = h.data_entrega
LEFT JOIN manut_veiculo mv ON fe.sk_veiculo = mv.veiculo_id AND mv.rn = 1 AND mv.data_conclusao <= fe.data_entrega;


In [0]:
%sql
SELECT COUNT(*) FROM vw_features_ml WHERE dias_ultima_manutencao IS NULL;

In [0]:
%sql
SELECT target_atraso, COUNT(*) AS volume, ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
FROM vw_features_ml
GROUP BY 1;


In [0]:
%sql
SELECT corr(distancia_km, target_atraso) AS correlacao_dist_target FROM vw_features_ml;